# Antardrishti — comparable scores, the rule as it runs, and region growing

**Three questions, in order.**

0. **What does the shipped radar rule score by the protocol papers use?** Every figure so far (IoU 0.489) pooled all 441 labelled chips. Published Sen1Floods11 results score the official *test* split, usually averaging IoU per chip, with Bolivia held out as an unseen region. Until we score that way, our numbers cannot sit in the same table as theirs.
1. **What does it score as it actually runs?** Validation used raw 10 m chips. The live app speckle-filters (50 m focal median) and then reduces at **100–200 m**, where Earth Engine serves block-averaged pixels. −20 dB was tuned on the first and is applied to the second.
2. **Does region growing recover the missed water?** Threshold only the confident water, then grow into touching pixels that pass a looser test (Matgen et al. 2011; Giustarini et al. 2013; DLR's algorithm in the Copernicus Global Flood Monitoring ensemble). 97% of our missed water is within a few dB of the threshold, which is exactly what this is built to reach.

**The rules.** Every setting is chosen on the **train** split only and reported on **test** and **Bolivia**, which the tuning never sees. A method earns a place only if **test IoU rises** over the darkness rule. Both averaging conventions are reported.

**Caveats that apply to every number here.** Hand labels start from a Sentinel-2 classification that analysts edited; pixels they could not identify confidently were marked no-data, so the hardest ground is never scored — by any method, ours or published.


In [1]:
# One-time install. scipy is already present if scikit-learn is installed.
!pip install rasterio numpy scipy


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data and the official split

Same folder as notebooks 05 and 06. The split files ship with the dataset (`splits/flood_handlabeled/flood_{train,valid,test,bolivia}_data.csv`).

In [2]:
LOCAL = r"C:\sen1floods11"

import sys
from pathlib import Path

root = Path(LOCAL)
if not root.exists():
    raise SystemExit(f"{root} does not exist")

EVALUATION = Path.cwd().resolve().parent.parent / "evaluation"
sys.path.insert(0, str(EVALUATION))

import json, time
import numpy as np
import rasterio

import importlib
import spatial as SP
import change as C

# Reload on every run. Python keeps a module from its first import for the
# whole session, so after spatial.py changes a notebook that is re-run without
# restarting would still use the old code - which is exactly what happened
# with load_splits(hand_keys=...).
importlib.reload(SP)
importlib.reload(C)

s1 = {p.name.replace("_S1Hand.tif", ""): p for p in root.rglob("*_S1Hand.tif")}
labels = {p.name.replace("_LabelHand.tif", ""): p for p in root.rglob("*_LabelHand.tif")}

# Split files are recognised by content: only files whose chips have hand
# labels on disk count, which separates them from the weak-label splits.
splits = SP.load_splits(root, hand_keys=set(labels))
for name, source in SP.split_sources(root, hand_keys=set(labels)).items():
    print(f"{name:<8} read from {source}")
print()

for name in ("train", "valid", "test", "bolivia"):
    have = splits[name] & set(s1) & set(labels)
    print(f"{name:<8} {len(splits[name]):>4} listed, {len(have):>4} found on disk")

overlap = splits["train"] & splits["test"]
if overlap:
    raise SystemExit(f"train and test share chips: {sorted(overlap)[:5]}")

bolivia  read from C:\sen1floods11\sen1floods11_v1.1\splits\flood_bolivia_data.txt
test     read from C:\sen1floods11\sen1floods11_v1.1\splits\flood_test_data.txt
train    read from C:\sen1floods11\sen1floods11_v1.1\splits\flood_train_data.txt
valid    read from C:\sen1floods11\sen1floods11_v1.1\splits\flood_valid_data.txt

train     252 listed,  252 found on disk
valid      89 listed,   89 found on disk
test       90 listed,   90 found on disk
bolivia    15 listed,   15 found on disk


In [3]:
# Per chip: fused mean(VV,VH) in dB, raw and speckle-filtered as production
# does it, plus the labels. The speckle filter is the slow part - a few minutes.
LIMIT = None
split_of = {k: name for name in ("train", "valid", "test", "bolivia") for k in splits[name]}

def load(key):
    with rasterio.open(s1[key]) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
    vv[vv < C.NODATA_FLOOR_DB] = np.nan
    vh[vh < C.NODATA_FLOOR_DB] = np.nan
    with rasterio.open(labels[key]) as src:
        truth = src.read(1).astype(np.int8)
    return vv, vh, truth

chips, started = [], time.time()
keys = sorted(k for k in split_of if k in s1 and k in labels)
for i, key in enumerate(keys[:LIMIT] if LIMIT else keys, 1):
    vv, vh, truth = load(key)
    truth = truth.copy()
    truth[~(np.isfinite(vv) & np.isfinite(vh))] = -1     # scored only where radar has data
    if not (truth != -1).any():
        continue
    chips.append({
        "key": key, "split": split_of[key], "truth": truth,
        "fused": C.fused(vv, vh).astype(np.float32),
        "speckled": C.fused(SP.speckle_median(vv), SP.speckle_median(vh)).astype(np.float32),
    })
    if i % 50 == 0:
        print(f"  {i}/{len(keys)} chips, {time.time() - started:.0f}s")

by_split = {name: [c for c in chips if c["split"] == name] for name in ("train", "valid", "test", "bolivia")}
print({name: len(v) for name, v in by_split.items()}, f"loaded in {time.time() - started:.0f}s")

  50/446 chips, 25s
  100/446 chips, 45s
  150/446 chips, 67s
  200/446 chips, 88s
  250/446 chips, 109s
  300/446 chips, 130s
  350/446 chips, 151s
  400/446 chips, 171s
{'train': 251, 'valid': 86, 'test': 88, 'bolivia': 15} loaded in 190s


## 0. The shipped rule, scored the way papers score

For reference, published test-split figures (averaged per chip, their conventions): VH threshold 0.359 (all water, Bonafilia et al. 2020); FCNN 0.493; U-Net on radar 0.45, with elevation 0.57 (Konapala et al. 2021); Attentive U-Net 0.672 (Yadav et al. 2022). Comparable in protocol, not identical in convention - read them as a neighbourhood, not a leaderboard.

In [4]:
def dark(db):
    return lambda c: np.isfinite(c["fused"]) & (c["fused"] < db)

def row(name, result):
    p = result["pooled"]
    return (f"{name:<44}{p['iou']:>8.3f}{result['chip_mean_iou']:>10.3f}"
            f"{p['precision']:>8.3f}{p['recall']:>8.3f}{result['chips']:>7}")

HEAD = f"{'':<44}{'pooled':>8}{'chip-mean':>10}{'prec':>8}{'recall':>8}{'chips':>7}"

print("shipped rule, mean(VV,VH) < -20 dB, raw 10 m\n" + HEAD)
shipped = {}
for name in ("train", "valid", "test", "bolivia"):
    shipped[name] = SP.score(by_split[name], dark(-20.0))
    print(row(name, shipped[name]))
shipped["all"] = SP.score(chips, dark(-20.0))
print(row("all chips (our previous protocol)", shipped["all"]))

# -20 dB was chosen with the test chips included. A fair baseline picks the
# threshold on train only.
sweep_dark = {db: SP.score(by_split["train"], dark(db))["pooled"]["iou"]
              for db in np.arange(-24.0, -14.5, 0.5)}
DARK_TRAIN = max(sweep_dark, key=sweep_dark.get)
print(f"\nthreshold chosen on train only: {DARK_TRAIN:.1f} dB (train IoU {sweep_dark[DARK_TRAIN]:.3f})")
fair = {name: SP.score(by_split[name], dark(DARK_TRAIN)) for name in ("test", "bolivia")}
print(row(f"test, {DARK_TRAIN:.1f} dB (train-chosen)", fair["test"]))
print(row(f"bolivia, {DARK_TRAIN:.1f} dB (train-chosen)", fair["bolivia"]))

shipped rule, mean(VV,VH) < -20 dB, raw 10 m
                                              pooled chip-mean    prec  recall  chips
train                                          0.471     0.261   0.738   0.566    251
valid                                          0.504     0.288   0.788   0.582     86
test                                           0.515     0.295   0.795   0.594     88
bolivia                                        0.553     0.353   0.903   0.588     15
all chips (our previous protocol)              0.491     0.276   0.768   0.577    440

threshold chosen on train only: -19.5 dB (train IoU 0.472)
test, -19.5 dB (train-chosen)                  0.522     0.300   0.765   0.622     88
bolivia, -19.5 dB (train-chosen)               0.597     0.382   0.892   0.643     15


## 1. The rule as it actually runs

Speckle filter first, then the analysis scale. At 100 m Earth Engine serves the mean of each 10 × 10 block; at 200 m, of each 20 × 20. Labels are aggregated by majority, and a coarse pixel is scored only if most of it was labelled.

In [5]:
def coarse_chips(split, factor, source="fused"):
    return [{"truth": SP.block_label(c["truth"], factor),
             "fused": SP.block_mean(c[source], factor)} for c in by_split[split]]

print(HEAD)
print(row("test, raw 10 m", shipped["test"]))
as_runs = {"speckled_10m": SP.score(by_split["test"], lambda c: np.isfinite(c["speckled"]) & (c["speckled"] < -20.0))}
print(row("test, speckle-filtered 10 m", as_runs["speckled_10m"]))

scale_best = {}
for factor, label in ((10, "100 m"), (20, "200 m")):
    test_coarse = coarse_chips("test", factor)
    as_runs[label] = SP.score(test_coarse, dark(-20.0))
    print(row(f"test, {label} (production scale)", as_runs[label]))

    # Is -20 dB still right at this scale? Chosen on train, scored on test.
    train_coarse = coarse_chips("train", factor)
    sweep = {db: SP.score(train_coarse, dark(db))["pooled"]["iou"] for db in np.arange(-24.0, -14.5, 0.5)}
    best = max(sweep, key=sweep.get)
    scale_best[label] = {"db": float(best), "test": SP.score(test_coarse, dark(best))}
    print(row(f"test, {label} at {best:.1f} dB (train-chosen)", scale_best[label]["test"]))

print("\nIf the best threshold at production scale differs from -20 dB, the")
print("shipped rule is tuned for a resolution it never runs at.")

                                              pooled chip-mean    prec  recall  chips
test, raw 10 m                                 0.515     0.295   0.795   0.594     88
test, speckle-filtered 10 m                    0.509     0.290   0.860   0.555     88
test, 100 m (production scale)                 0.529     0.337   0.855   0.581     88
test, 100 m at -19.0 dB (train-chosen)         0.570     0.349   0.808   0.659     88
test, 200 m (production scale)                 0.524     0.339   0.879   0.565     88
test, 200 m at -18.5 dB (train-chosen)         0.609     0.377   0.807   0.713     88

If the best threshold at production scale differs from -20 dB, the
shipped rule is tuned for a resolution it never runs at.


## 2. Region growing, tuned on train

Seeds below −20 dB (the shipped rule). Then grow, `steps` times, into any touching pixel below the weak threshold. `steps = 0` grows until nothing changes. Pixels are 10 m, so `steps = 10` reaches about 100 m from confident water.

In [6]:
weaks = [-19.0, -18.0, -17.0, -16.0, -15.0, -14.0]
steps_list = [2, 5, 10, 20, 0]

def grower(weak, steps, source="fused"):
    return lambda c: SP.grow(c[source], -20.0, weak, steps)

grid = {}
started = time.time()
for weak in weaks:
    for steps in steps_list:
        grid[(weak, steps)] = SP.score(by_split["train"], grower(weak, steps))["pooled"]
print(f"swept in {time.time() - started:.0f}s\n")

print("train IoU     " + "".join(f"{('all' if s == 0 else s):>8}" for s in steps_list) + "   <- steps")
for weak in weaks:
    print(f"weak {weak:>5.1f}  " + "".join(f"{grid[(weak, s)]['iou']:>8.3f}" for s in steps_list))
BEST_GROW = max(grid, key=lambda k: grid[k]["iou"])
print(f"\nbest on train: weak {BEST_GROW[0]} dB, steps {BEST_GROW[1] or 'until stable'}, "
      f"IoU {grid[BEST_GROW]['iou']:.3f} (darkness alone on train: {sweep_dark[-20.0]:.3f})")

swept in 30s

train IoU            2       5      10      20     all   <- steps
weak -19.0     0.474   0.473   0.473   0.473   0.473
weak -18.0     0.467   0.461   0.460   0.460   0.460
weak -17.0     0.457   0.438   0.432   0.431   0.431
weak -16.0     0.446   0.409   0.393   0.387   0.385
weak -15.0     0.439   0.382   0.350   0.333   0.327
weak -14.0     0.434   0.359   0.311   0.279   0.257

best on train: weak -19.0 dB, steps 2, IoU 0.474 (darkness alone on train: 0.471)


In [7]:
weak, steps = BEST_GROW
results = {}
for split in ("test", "bolivia"):
    results[split] = {
        "darkness -20 dB (ships)": shipped[split],
        f"darkness {DARK_TRAIN:.1f} dB (train-chosen)": fair[split],
        f"grow, weak {weak}, steps {steps or 'all'}": SP.score(by_split[split], grower(weak, steps)),
        f"grow on speckle-filtered": SP.score(by_split[split], grower(weak, steps, "speckled")),
    }
    print(f"\n{split.upper()} - never seen by any tuning\n" + HEAD)
    for name, result in results[split].items():
        print(row(name, result))

base = results["test"]["darkness -20 dB (ships)"]["pooled"]["iou"]
grown = results["test"][f"grow, weak {weak}, steps {steps or 'all'}"]["pooled"]["iou"]
print(f"\ntest IoU change from region growing: {grown - base:+.3f}")
print("Region growing earns a place only if that number is positive - and a")
print("large drop on Bolivia would mean it does not travel to unseen regions.")


TEST - never seen by any tuning
                                              pooled chip-mean    prec  recall  chips
darkness -20 dB (ships)                        0.515     0.295   0.795   0.594     88
darkness -19.5 dB (train-chosen)               0.522     0.300   0.765   0.622     88
grow, weak -19.0, steps 2                      0.530     0.307   0.745   0.648     88
grow on speckle-filtered                       0.525     0.304   0.826   0.591     88

BOLIVIA - never seen by any tuning
                                              pooled chip-mean    prec  recall  chips
darkness -20 dB (ships)                        0.553     0.353   0.903   0.588     15
darkness -19.5 dB (train-chosen)               0.597     0.382   0.892   0.643     15
grow, weak -19.0, steps 2                      0.632     0.406   0.884   0.689     15
grow on speckle-filtered                       0.628     0.386   0.926   0.662     15

test IoU change from region growing: +0.015
Region growing earns a pla

In [8]:
# What it recovered, in notebook 06's bins, on the test split - and what it cost.
shipped_rule = dark(-20.0)
grow_rule = grower(weak, steps)
bins = {b: {"missed": 0, "recovered": 0} for b in ("marginal", "moderate", "bright")}
added_fp = 0
for c in by_split["test"]:
    base_pred = shipped_rule(c)
    new_pred = grow_rule(c)
    for b, v in C.recovery(c["fused"], c["truth"], base_pred, new_pred).items():
        bins[b]["missed"] += v["missed"]; bins[b]["recovered"] += v["recovered"]
    added_fp += C.added_false_positives(c["truth"], base_pred, new_pred)

print(f"{'bin':<10}{'missed':>12}{'recovered':>12}{'share':>9}")
for b, v in bins.items():
    print(f"{b:<10}{v['missed']:>12,}{v['recovered']:>12,}{(v['recovered'] / v['missed'] if v['missed'] else 0):>9.1%}")
recovered = sum(v["recovered"] for v in bins.values())
print(f"\nwater recovered: {recovered:,}   dry pixels newly called water: {added_fp:,}")
# Total mistakes and IoU can disagree: a recovered pixel turns a miss into a
# hit without enlarging IoU's denominator, a false alarm enlarges it. IoU rises
# whenever recovered / added exceeds the current IoU.
print(f"total mistakes change: {added_fp - recovered:+,} pixels")
print(f"recovered / added = {recovered / max(added_fp, 1):.2f}; IoU rises if this exceeds the current IoU")

bin             missed   recovered    share
marginal       434,460     134,338    30.9%
moderate       561,784           0     0.0%
bright          24,559           0     0.0%

water recovered: 134,338   dry pixels newly called water: 173,271


In [9]:
def slim(result):
    return {"pooled": result["pooled"], "chip_mean_iou": result["chip_mean_iou"], "chips": result["chips"]}

grown_test = results["test"][f"grow, weak {weak}, steps {steps or 'all'}"]
payload = {
    "protocol": "Sen1Floods11 official hand-labelled split; tuned on train only; "
                "reported on test and Bolivia; pooled IoU and per-chip mean IoU",
    "chips": {name: len(v) for name, v in by_split.items()},
    "step0_shipped_rule": {k: slim(v) for k, v in shipped.items()},
    "step0_train_chosen_threshold_db": float(DARK_TRAIN),
    "step0_train_chosen_on": {k: slim(v) for k, v in fair.items()},
    "step1_as_it_runs_on_test": {k: slim(v) for k, v in as_runs.items()},
    "step1_best_threshold_at_scale": {k: {"db": v["db"], "test": slim(v["test"])} for k, v in scale_best.items()},
    "step2_grow_grid_train_iou": {f"{w}|{s}": grid[(w, s)]["iou"] for w, s in grid},
    "step2_chosen": {"strong_db": -20.0, "weak_db": weak, "steps": steps},
    "step2_results": {split: {k: slim(v) for k, v in r.items()} for split, r in results.items()},
    "step2_recovery_on_test": bins,
    "step2_added_false_positives_on_test": added_fp,
    "earns_a_place": grown_test["pooled"]["iou"] > shipped["test"]["pooled"]["iou"],
    "validation_block_if_it_does": {
        "dataset": f"Sen1Floods11 HandLabeled test split ({len(by_split['test'])} chips), tuned on train",
        "iou": round(grown_test["pooled"]["iou"], 3),
        "precision": round(grown_test["pooled"]["precision"], 3),
        "recall": round(grown_test["pooled"]["recall"], 3),
    },
}
out = EVALUATION / "spatial_results.json"
out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"written to {out}")
print("earns a place:", payload["earns_a_place"])

written to C:\Users\punit\OneDrive\Desktop\Group_project\application\antardrishti\evaluation\spatial_results.json
earns a place: True


In [10]:
for split in ("valid", "bolivia"):
    for factor, dbs in ((10, (-20.0, -19.0)), (20, (-20.0, -18.5))):
        cc = coarse_chips(split, factor)
        for db in dbs:
            print(row(f"{split}, {factor * 10} m at {db} dB", SP.score(cc, dark(db))))

valid, 100 m at -20.0 dB                       0.538     0.323   0.872   0.585     86
valid, 100 m at -19.0 dB                       0.564     0.324   0.807   0.653     86
valid, 200 m at -20.0 dB                       0.544     0.358   0.901   0.578     86
valid, 200 m at -18.5 dB                       0.594     0.387   0.801   0.696     86
bolivia, 100 m at -20.0 dB                     0.596     0.391   0.945   0.618     15
bolivia, 100 m at -19.0 dB                     0.682     0.458   0.919   0.725     15
bolivia, 200 m at -20.0 dB                     0.579     0.484   0.961   0.594     15
bolivia, 200 m at -18.5 dB                     0.718     0.593   0.913   0.770     15


## What to do with the numbers

**Step 0 is a result on its own.** The shipped rule's test-split score, in both conventions, is the figure to put beside published ones in Chapter 7 — not 0.489, which pooled every chip including the ones the threshold was tuned on.

**Step 1 decides whether the threshold is right where it runs.** If the train-chosen threshold at 100 m or 200 m differs clearly from −20 dB, the backend is applying a 10 m threshold to 200 m pixels. That is a one-constant fix, but only with the measurement behind it — paste the step-1 result back and we change it together.

**Step 2 earns a place only if `earns_a_place` is true.** If it is, region growing goes into `backend/detection/sar.py` as an Earth Engine geodesic dilation (repeated one-pixel focal max, masked by the weak test) — the same operation scored here. Note the pixel-size issue: `steps` counts 10 m pixels here, so it has to be converted to metres for the 100–200 m scale the backend runs at. If it is false, say so in Chapter 7 with the recovery table: which missed water growing could and could not reach.
